# 04 — Weather-Aware Model + Final Comparison

XGBoost with lag features, rolling averages, calendar features, plus
temperature and temperature deviation from the historical norm for that day
of year. This is the model expected to close the extreme-day gap found in
notebook 03.

It also fits a no-weather ablation (same model, same lag/rolling/calendar
features, temperature removed) so the weather effect can be isolated from
the effect of just switching from SARIMAX to XGBoost.

This notebook builds the single comparison table worth screenshotting
for a resume or interview: baseline, ablation, and weather-aware, normal
days vs extreme days, side by side.

In [ ]:
# lets src/ be imported when running this notebook from the notebooks/ folder
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import shap

from src import config, utils, weather_model

y_test, predicted, model, X_test = weather_model.run_weather_model()

## Predicted vs actual — same two-week window as the baseline notebook

In [ ]:
window = slice(0, 24 * 14)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y_test.index[window], y_test.iloc[window], label="actual", linewidth=1)
ax.plot(predicted.index[window], predicted.iloc[window], label="predicted", linewidth=1)
ax.legend()
ax.set_title("Weather-aware model — first two weeks of test year")
ax.set_ylabel("MW")
plt.show()

## SHAP feature importance

Confirms *why* the model improved, not just that it did. If temperature and
temperature deviation don't show up meaningfully here, the improvement isn't
provably about weather, and that claim should not be made in the write-up.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test, show=True)

## Extreme-day segmentation for the weather-aware model

Same extreme-day dates from notebook 03, applied here for a fair comparison.

In [ ]:
df = pd.read_parquet(config.JOINED_DATA_PATH).asfreq("h")
train, test = utils.time_ordered_split(df)
thresholds = utils.compute_extreme_thresholds(utils.daily_min_max_temp(train))
extreme_flags = utils.flag_extreme_days(utils.daily_min_max_temp(test), thresholds)
extreme_dates = set(extreme_flags[extreme_flags].index.date)

weather_preds = pd.read_parquet(config.WEATHER_MODEL_PREDICTIONS_PATH)
weather_result = utils.segment_metrics_by_extreme_day(weather_preds, extreme_dates)
weather_result

## No-weather ablation

The SHAP plot above shows what the weather-aware model leans on, but it doesn't
by itself prove weather is what closes the extreme-day gap versus the baseline —
the baseline and this model differ in more than one way (SARIMAX vs XGBoost,
and this model also gets real recent-load lag features the baseline never sees).

This cell fits the exact same XGBoost, same lag/rolling/calendar features,
with `temp_c` and `temp_deviation` removed. Same model class, same lag horizon,
weather is the only thing that changes — so comparing *this* against the
weather-aware model isolates what weather specifically contributes.

In [ ]:
y_test_nw, predicted_nw, model_nw, X_test_nw = weather_model.run_weather_model(use_weather=False)

In [ ]:
no_weather_preds = pd.read_parquet(config.NO_WEATHER_MODEL_PREDICTIONS_PATH)
no_weather_result = utils.segment_metrics_by_extreme_day(no_weather_preds, extreme_dates)
no_weather_result

## The comparison table

Baseline vs weather-aware, normal vs extreme days, in one place.

In [ ]:
baseline_preds = pd.read_parquet(config.BASELINE_PREDICTIONS_PATH)
baseline_result = utils.segment_metrics_by_extreme_day(baseline_preds, extreme_dates)

comparison = pd.DataFrame([
    {"model": "baseline (calendar only, SARIMAX)", **baseline_result},
    {"model": "no-weather ablation (lag+calendar, XGBoost)", **no_weather_result},
    {"model": "weather-aware (lag+calendar+temp, XGBoost)", **weather_result},
]).set_index("model")

comparison

In [ ]:
config.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(config.COMPARISON_TABLE_PATH)
print(f"saved to {config.COMPARISON_TABLE_PATH}")

## Reading the table

- **Extreme-day gap closed (vs. baseline)** = `baseline extreme_mape - weather extreme_mape`.
  This is the headline number, but it conflates two changes at once: XGBoost vs.
  SARIMAX, and weather vs. no weather.
- **Extreme-day gap closed (from weather specifically)** = `no_weather ablation
  extreme_mape - weather extreme_mape`. Same model class, same lag features on both
  sides — this is the number that actually isolates what temperature contributed,
  and the one to lead with if asked "how do you know it was the weather."
- Compare `weather extreme_mape` to `weather normal_mape` too — a well-built
  weather-aware model should show a much smaller gap between normal and
  extreme days than the baseline did, that's the real proof the temperature
  features are doing their job, not just that the model got better overall.
- If the ablation and the weather-aware model land close together, that's a real
  result too — it means the lag/calendar features were already capturing most of
  what temperature would add, and the honest write-up says that rather than
  overstating weather's role.